# 1D-CNN Model Architecture and Training

In [4]:
import torch

if torch.backends.mps.is_available():
    device = torch.device("mps")
    print("MPS device found.")
else:
    print("MPS device not found.")

MPS device found.


In [12]:
from form_analyzer_model import FormAnalyzer1DCNN

In [13]:
keypoint_path = "datasets/custom_running_form/keypoints"

def train_model(model, optimizer, loss_fn, train_loader, val_loader, num_epochs=20):
    # Loss function for Multi-Label Classification
    model_save_path = "runs/form/weights/form_analyzer_model.pt"
    
    best_val_loss = float('inf')
    
    for epoch in range(num_epochs):
        model.train()
        running_train_loss = 0.0
        
        for features, labels in train_loader:
            features, labels = features.to(device), labels.to(device)

            optimizer.zero_grad()
            
            outputs = model(features)
            
            loss = loss_fn(outputs, labels)

            loss.backward()
            
            optimizer.step()
            
            running_train_loss += loss.item() * features.size(0)
            
        epoch_train_loss = running_train_loss / len(train_loader.dataset)
        

        model.eval()
        running_val_loss = 0.0
        all_preds = []
        all_labels = []
        
        # Disable gradient tracking for evaluation to save memory and speed up computation
        with torch.no_grad():
            for features, labels in val_loader:
                features, labels = features.to(device), labels.to(device)
                
                outputs = model(features)
                loss = loss_fn(outputs, labels)
                running_val_loss += loss.item() * features.size(0)
                
                probs = torch.sigmoid(outputs)
                
                preds = (probs > 0.5).float()
                
                all_preds.append(preds.cpu())
                all_labels.append(labels.cpu())
                
        epoch_val_loss = running_val_loss / len(val_loader.dataset)
        
        # Calculate Exact Match Accuracy 
        # (For a sample to be "correct", all 4 labels must match exactly)
        all_preds = torch.cat(all_preds)
        all_labels = torch.cat(all_labels)
        exact_match_acc = (all_preds == all_labels).all(dim=1).float().mean().item()
        
        print(f"Epoch [{epoch+1:02d}/{num_epochs:02d}] | Train Loss: {epoch_train_loss:.4f} | Val Loss: {epoch_val_loss:.4f} | Val Exact Match Acc: {exact_match_acc * 100:.2f}%")
        
        if epoch_val_loss < best_val_loss:
            best_val_loss = epoch_val_loss
            torch.save(model.state_dict(), model_save_path)
            
    print("Training complete.")
    return model

In [8]:
from form_analyzer_helper import create_dataloaders

train_loader, val_loader, test_loader = create_dataloaders("datasets/custom_running_form/custom_labels.csv")

Dataset Split -> Train: 331 | Val: 71 | Test: 72


In [14]:
import torch.nn as nn
import torch.optim as optim

learning_rate = 0.001

model = FormAnalyzer1DCNN().to(device)

optimizer = optim.Adam(model.parameters(), lr=learning_rate)
loss_fn = nn.BCEWithLogitsLoss()

In [16]:
train_model(model, optimizer, loss_fn, train_loader, val_loader, num_epochs=20)

Epoch [01/20] | Train Loss: 0.0749 | Val Loss: 0.0676 | Val Exact Match Acc: 94.37%
Epoch [02/20] | Train Loss: 0.0827 | Val Loss: 0.2300 | Val Exact Match Acc: 67.61%
Epoch [03/20] | Train Loss: 0.0656 | Val Loss: 0.3236 | Val Exact Match Acc: 67.61%
Epoch [04/20] | Train Loss: 0.0902 | Val Loss: 0.2299 | Val Exact Match Acc: 56.34%
Epoch [05/20] | Train Loss: 0.1265 | Val Loss: 0.2801 | Val Exact Match Acc: 76.06%
Epoch [06/20] | Train Loss: 0.0981 | Val Loss: 0.1254 | Val Exact Match Acc: 76.06%
Epoch [07/20] | Train Loss: 0.0641 | Val Loss: 0.0502 | Val Exact Match Acc: 92.96%
Epoch [08/20] | Train Loss: 0.0796 | Val Loss: 0.1630 | Val Exact Match Acc: 88.73%
Epoch [09/20] | Train Loss: 0.0663 | Val Loss: 0.4752 | Val Exact Match Acc: 42.25%
Epoch [10/20] | Train Loss: 0.0483 | Val Loss: 0.3355 | Val Exact Match Acc: 67.61%
Epoch [11/20] | Train Loss: 0.0565 | Val Loss: 0.3800 | Val Exact Match Acc: 73.24%
Epoch [12/20] | Train Loss: 0.0557 | Val Loss: 0.1785 | Val Exact Match Acc:

FormAnalyzer1DCNN(
  (conv_block): Sequential(
    (0): Conv1d(51, 64, kernel_size=(3,), stride=(1,), padding=(1,))
    (1): ReLU()
    (2): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (3): Dropout(p=0.3, inplace=False)
    (4): Conv1d(64, 128, kernel_size=(3,), stride=(1,), padding=(1,))
    (5): ReLU()
    (6): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (7): Dropout(p=0.3, inplace=False)
  )
  (global_pool): AdaptiveAvgPool1d(output_size=1)
  (classifier): Sequential(
    (0): Linear(in_features=128, out_features=64, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.3, inplace=False)
    (3): Linear(in_features=64, out_features=4, bias=True)
  )
)